In [3]:
# ============================================
# STEP 1: Install libraries
# ============================================
# !pip install requests

# ============================================
# STEP 2: API Key
# ============================================
RAPIDAPI_KEY = "8a9723f9b2mshe2af1b3c9a98d67p16ad04jsn3951e009ff18"

# ============================================
# STEP 3: Search city to get destination ID
# ============================================
import requests
from datetime import datetime

def search_destination(city):
    print(f"\n🔍 Searching destination for: {city}")

    url = "https://booking-com15.p.rapidapi.com/api/v1/hotels/searchDestination"

    headers = {
        "X-RapidAPI-Key": RAPIDAPI_KEY,
        "X-RapidAPI-Host": "booking-com15.p.rapidapi.com"
    }

    params = {"query": city}

    response = requests.get(url, headers=headers, params=params)
    data = response.json()

    if not data['status']:
        print("❌ City not found!")
        return None, None

    for item in data['data']:
        if item['search_type'] == 'city':
            print(f"✅ Found: {item['city_name']}, {item['country']}")
            print(f"   Total hotels available: {item['nr_hotels']}")
            return item['dest_id'], item['dest_type']

    return None, None


# ============================================
# STEP 4: Search Hotels
# ============================================
def search_hotels(dest_id, dest_type, check_in, check_out, adults=2, children=0):
    print(f"\n🏨 Searching hotels...")
    print(f"   Check-in:  {check_in}")
    print(f"   Check-out: {check_out}")
    print(f"   Adults:    {adults}")
    print(f"   Children:  {children}")

    url = "https://booking-com15.p.rapidapi.com/api/v1/hotels/searchHotels"

    headers = {
        "X-RapidAPI-Key": RAPIDAPI_KEY,
        "X-RapidAPI-Host": "booking-com15.p.rapidapi.com"
    }

    params = {
        "dest_id": dest_id,
        "search_type": dest_type,
        "arrival_date": check_in,
        "departure_date": check_out,
        "adults": adults,
        "children_age": ",".join(["5"] * children) if children > 0 else None,  # default child age = 5
        "room_qty": 1,
        "page_number": 1,
        "units": "metric",
        "temperature_unit": "c",
        "languagecode": "en-us",
        "currency_code": "INR"
    }

    # Remove None values
    params = {k: v for k, v in params.items() if v is not None}

    response = requests.get(url, headers=headers, params=params)
    data = response.json()

    if not data['status']:
        print("❌ No hotels found!")
        return []

    hotels = data['data']['hotels']
    print(f"✅ Found {len(hotels)} hotels!\n")
    return hotels


# ============================================
# STEP 5: Display Hotels Nicely
# ============================================
def display_hotels(hotels):
    print("=" * 60)
    print("🏨 AVAILABLE HOTELS")
    print("=" * 60)

    hotel_list = []

    for i, hotel in enumerate(hotels[:10]):
        prop = hotel['property']

        name    = prop.get('name', 'Unknown Hotel')
        rating  = prop.get('reviewScore', 0)
        reviews = prop.get('reviewCount', 0)
        price   = prop.get('priceBreakdown', {}).get('grossPrice', {}).get('value', 0)
        country = prop.get('countryCode', 'in')

        booking_url = f"https://www.booking.com/hotel/{country}/{name.lower().replace(' ', '-')}.html"

        hotel_list.append({
            'index': i + 1,
            'name': name,
            'rating': rating,
            'reviews': reviews,
            'price_per_night': round(price),
            'booking_url': booking_url
        })

        print(f"\n[{i+1}] {name}")
        print(f"    ⭐ Rating:   {rating}/10 ({reviews} reviews)")
        print(f"    💰 Price:    ₹{round(price):,} per night")
        print(f"    🔗 Book URL: {booking_url}")

    print("\n" + "=" * 60)
    return hotel_list


# ============================================
# STEP 6: Get Hotel Details
# ============================================
def get_hotel_details(hotel_id):
    print(f"\n📋 Getting details for hotel ID: {hotel_id}")

    url = "https://booking-com15.p.rapidapi.com/api/v1/hotels/getHotelDetails"

    headers = {
        "X-RapidAPI-Key": RAPIDAPI_KEY,
        "X-RapidAPI-Host": "booking-com15.p.rapidapi.com"
    }

    params = {
        "hotel_id": hotel_id,
        "languagecode": "en-us"
    }

    response = requests.get(url, headers=headers, params=params)
    data = response.json()

    if data.get('status'):
        details = data.get('data', {})
        print(f"✅ Hotel: {details.get('hotel_name', 'N/A')}")
        print(f"   Address: {details.get('address', 'N/A')}")
        print(f"   City: {details.get('city', 'N/A')}")
        print(f"   Stars: {'⭐' * int(details.get('stars', 0))}")
        print(f"   Description: {details.get('description_translations', [{}])[0].get('description', 'N/A')[:200]}...")
        return details

    return None


# ============================================
# STEP 7: Show Booking Link
# ============================================
def book_hotel(hotel):
    print("\n" + "=" * 60)
    print("🎉 BOOKING DETAILS")
    print("=" * 60)
    print(f"\n✅ Hotel Selected: {hotel['name']}")
    print(f"⭐ Rating: {hotel['rating']}/10")
    print(f"💰 Price: ₹{hotel['price_per_night']:,} per night")
    print(f"\n📌 NOTE: This API searches hotels only.")
    print(f"   Actual booking happens on Booking.com website.")
    print(f"\n🔗 Click this link to complete your booking:")
    print(f"\n   👉 {hotel['booking_url']}\n")
    print("=" * 60)


# ============================================
# STEP 8: GET USER INPUTS
# ============================================

def get_valid_date(prompt):
    while True:
        date_str = input(prompt).strip()
        try:
            datetime.strptime(date_str, "%Y-%m-%d")
            return date_str
        except ValueError:
            print("   ⚠️  Invalid format. Please use YYYY-MM-DD (e.g. 2026-05-10)")

def get_valid_int(prompt, min_val=0, max_val=20):
    while True:
        try:
            val = int(input(prompt).strip())
            if min_val <= val <= max_val:
                return val
            else:
                print(f"   ⚠️  Please enter a number between {min_val} and {max_val}")
        except ValueError:
            print("   ⚠️  Please enter a valid number")

print("=" * 60)
print("🏨 HOTEL SEARCH - BANGALORE")
print("=" * 60)

CITY      = "Bangalore"
CHECK_IN  = get_valid_date("📅 Enter Check-in date  (YYYY-MM-DD): ")
CHECK_OUT = get_valid_date("📅 Enter Check-out date (YYYY-MM-DD): ")

# Validate check-out is after check-in
while CHECK_OUT <= CHECK_IN:
    print("   ⚠️  Check-out must be after check-in!")
    CHECK_OUT = get_valid_date("📅 Enter Check-out date (YYYY-MM-DD): ")

ADULTS   = get_valid_int("👤 Enter number of Adults   (min 1): ", min_val=1, max_val=20)
CHILDREN = get_valid_int("👶 Enter number of Children (min 0): ", min_val=0, max_val=10)

nights = (datetime.strptime(CHECK_OUT, "%Y-%m-%d") - datetime.strptime(CHECK_IN, "%Y-%m-%d")).days
print(f"\n✅ Searching for {nights} night(s) | {ADULTS} adult(s) | {CHILDREN} child(ren)")


# ============================================
# STEP 9: RUN EVERYTHING
# ============================================

dest_id, dest_type = search_destination(CITY)

if dest_id:
    hotels = search_hotels(dest_id, dest_type, CHECK_IN, CHECK_OUT, ADULTS, CHILDREN)

    if hotels:
        hotel_list = display_hotels(hotels)

        best_hotel = max(hotel_list, key=lambda x: x['rating'])

        print(f"\n🏆 Best Rated Hotel: {best_hotel['name']}")
        print(f"   Rating: {best_hotel['rating']}/10")
        print(f"   Price:  ₹{best_hotel['price_per_night']:,}/night")
        print(f"   Total for {nights} night(s): ₹{best_hotel['price_per_night'] * nights:,}")

        book_hotel(best_hotel)

🏨 HOTEL SEARCH - BANGALORE
📅 Enter Check-in date  (YYYY-MM-DD): 2026-05-13
📅 Enter Check-out date (YYYY-MM-DD): 2026-05-18
👤 Enter number of Adults   (min 1): 2
👶 Enter number of Children (min 0): 0

✅ Searching for 5 night(s) | 2 adult(s) | 0 child(ren)

🔍 Searching destination for: Bangalore
✅ Found: Bangalore, India
   Total hotels available: 3237

🏨 Searching hotels...
   Check-in:  2026-05-13
   Check-out: 2026-05-18
   Adults:    2
   Children:  0
✅ Found 20 hotels!

🏨 AVAILABLE HOTELS

[1] Treebo Tiba
    ⭐ Rating:   7.2/10 (368 reviews)
    💰 Price:    ₹11,264 per night
    🔗 Book URL: https://www.booking.com/hotel/in/treebo-tiba.html

[2] Spektrum Hotel - Bengaluru Airport
    ⭐ Rating:   8.9/10 (473 reviews)
    💰 Price:    ₹20,079 per night
    🔗 Book URL: https://www.booking.com/hotel/in/spektrum-hotel---bengaluru-airport.html

[3] Hotel Stay Bliss HSR Layout
    ⭐ Rating:   8.4/10 (25 reviews)
    💰 Price:    ₹7,196 per night
    🔗 Book URL: https://www.booking.com/hotel/i